In [13]:
import requests
import base64

# 百度云控制台获取的 API Key 和 Secret Key
API_KEY = "6qMzbho0oyteWCkgIweRv0bc"
SECRET_KEY = "aUrcJXjfjCUHufIq9akwYtHnt7Dj99CR"

def get_access_token():
    """获取 Access Token"""
    url = f"https://aip.baidubce.com/oauth/2.0/token?grant_type=client_credentials&client_id={API_KEY}&client_secret={SECRET_KEY}"
    response = requests.get(url)
    return response.json().get("access_token") #向百度索取一个东西Token 获取数据

def ocr_high_precision(file_path):
    """
    百度高精度 OCR 识别
    支持 PDF 和图片
    """
    token = get_access_token()
    # 百度高精度接口地址
    request_url = "https://aip.baidubce.com/rest/2.0/ocr/v1/accurate_basic"
    
    with open(file_path, "rb") as f:
        img = base64.b64encode(f.read())#sajdasjkskdsj

    params = {"image": img}
    headers = {'content-type': 'application/x-www-form-urlencoded'}
    
    response = requests.post(f"{request_url}?access_token={token}", data=params, headers=headers) #发送数据
    
    if response.status_code == 200:#状态码
        # 拼接识别出的每一行文字
        result = response.json()
        full_text = "\n".join([line['words'] for line in result.get('words_result', [])])
        return full_text
    return None

# 使用示例
text = ocr_high_precision("/Users/roy/Library/CloudStorage/OneDrive-stu.haut.edu.cn/桌面/兼职/EasyEdu/homework/images (1).jpeg")
print(text)

2.(2019浙江宁波三模)已知函数)=
则函
数x)的零点为()
B.-2.0
D.0
【答案】D
【解析】当x≤1时，由x)=2-1=0,解得x=0:当x>1时，
由(x)=1+logx=0,解得x=,因为x>1,所以此时方程无解，综
上函数(x)的零点只有0.


In [10]:
import re
import json
import base64

def robust_parse_to_json(raw_text, chapter="01"):
    """
    重新设计的鲁棒性解析函数
    支持：1-2位题号、自动拼接换行题干、兼容【答案】/【解析】标识符
    """
    # 预处理：统一中文标点，去掉冗余空格
    text = raw_text.replace('：', ':').replace('（', '(').replace('）', ')')
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    if not lines:
        return []

    questions = []
    current_q = None

    # 1. 识别题号的正则 (支持 "2." 或 "02." 或 "2 .")
    num_pattern = re.compile(r'^(\d{1,2})\s*\.\s*(.*)', re.S)

    for line in lines:
        match = num_pattern.match(line)
        if match:
            # 如果匹配到新题号，先保存上一个题目
            if current_q:
                questions.append(current_q)
            
            # 初始化新题目字典
            current_q = {
                "raw_num": match.group(1),
                "full_text": match.group(2),
                "chapter": chapter
            }
        else:
            # 如果没匹配到题号，说明是上一题的换行内容，进行拼接
            if current_q:
                current_q["full_text"] += "\n" + line

    # 加入最后一个题目
    if current_q:
        questions.append(current_q)

    # 2. 对每个题目进行精细化字段切割
    final_data = []
    for q in questions:
        full_content = q["full_text"]
        
        # 利用【答案】或【解析】或 答案: 作为切割点
        # 即使 OCR 漏掉了括号，也能通过关键词切割
        parts = re.split(r'【?答案】?[:：]?|【?解析】?[:：]?', full_content)
        
        # 题干与选项（通常在第一部分）
        body_and_options = parts[0].strip()
        
        # 尝试从 body 中分离题干和选项 (匹配 A.)
        content_split = re.split(r'(?=A\.)', body_and_options, 1)
        content = content_split[0].strip()
        options = content_split[1].strip() if len(content_split) > 1 else "见题干"

        # 答案提取
        ans = "待校对"
        if len(parts) > 1:
            ans_match = re.search(r'([A-D])', parts[1])
            if ans_match:
                ans = ans_match.group(1)

        # 解析提取
        exp = parts[2].strip() if len(parts) > 2 else options

        # 生成 EasyEdu 格式
        item = {
            "id": f"ch{chapter}_{q['raw_num'].zfill(2)}",
            "title": content[:15].replace('\n', '') + "...", # 动态生成标题
            "content": content,
            "difficulty": "normal",
            "type": "multiple-choice",
            "reference_answer": {
                "content": ans,
                "explanation": exp
            },
            "chapter": chapter
        }
        final_data.append(item)
    
    return final_data

# 测试复杂数学题数据
test_text = """
2.(2019浙江宁波三模)已知函数)=\n则函\n数x)的零点为()\nA.2.0 B.-2.0\nD.0\n【答案】D\n【解析】当x≤1时，由x)=2-1=0,解得x=0:当x>1时，\n由(x)=1+logx=0,解得x=1/e...
"""

parsed_results = robust_parse_to_json(test_text)
print(json.dumps(parsed_results, indent=4, ensure_ascii=False))

[
    {
        "id": "ch01_02",
        "title": "(2019浙江宁波三模)已知函...",
        "content": "(2019浙江宁波三模)已知函数)=\n则函\n数x)的零点为()",
        "difficulty": "normal",
        "type": "multiple-choice",
        "reference_answer": {
            "content": "D",
            "explanation": "当x≤1时，由x)=2-1=0,解得x=0:当x>1时，\n由(x)=1+logx=0,解得x=1/e..."
        },
        "chapter": "01"
    }
]


In [11]:
import json
import re

def parse_text_to_json(text, chapter="01"):
    """
    逻辑：利用正则表达式从乱序文本中提取 题目、选项、答案
    """
    # 1. 提取标题 (假设第一行是标题)
    lines = text.strip().split('\n')
    title = lines[0] if lines else "未命名题目"
    
    # 2. 提取题目正文 (从标题后到选项前的内容)
    # 正则逻辑：匹配 A. 之前的所有内容
    # content_match = re.search(r'(.+?)(?=A\.)', text, re.S)
    # content = content_match.group(1).strip() if content_match else text
    
    # 3. 提取选项内容作为解析 (匹配 A. 到 答案 之间的所有内容)
    # 逻辑：找 A. 开始，直到“答案”或者“解析”关键字出现之前的所有文字
    options_match = re.search(r'(A\..+?)(?=答案|解析|$)', text, re.S)
    options_text = options_match.group(1).strip() if options_match else "通过 OCR 自动解析生成，请结合知识点校对。"

    # 3. 提取答案 (简单演示：查找“答案：”后面的字母)
    answer_match = re.search(r'答案[：:]\s*([A-D])', text)
    answer_content = answer_match.group(1) if answer_match else "待校对"

    # 4. 组装成 EasyEdu 标准格式
    question_json = {
        "id": f"q{chapter}{int(base64.b64encode(title[:5].encode()).hex()[:4], 16)}", # 模拟生成唯一ID
        "title": title,
        "content": content,
        "difficulty": "normal", # 默认难度
        "type": "concept",
        "reference_answer": {
            "content": answer_content,
            "explanation": options_text
        },
        "chapter": chapter
    }
    return question_json

In [12]:
import os
# C:\\Desktop
#/Users/roy/Downloads/本科成绩单.pdf
def batch_process_ocr(folder_path, output_json):
    all_questions = []
    
    # 逻辑：遍历文件夹内所有图片
    for file_name in os.listdir(folder_path):
        if file_name.endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(folder_path, file_name)
            print(f"正在识别: {file_name}...")
            
            # 1. 调用你之前的 OCR 函数获取文本
            raw_text = ocr_high_precision(full_path)
            
            # 2. 解析文本为字典对象
            if raw_text:
                question_data = parse_text_to_json(raw_text)
                all_questions.append(question_data)
    
    # 3. 批量持久化：一次性写入 JSON 文件
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(all_questions, f, ensure_ascii=False, indent=4)
    print(f"✅ 批量处理完成！已生成: {output_json}")

# 使用示例
batch_process_ocr("/Users/roy/Library/CloudStorage/OneDrive-stu.haut.edu.cn/桌面/兼职/EasyEdu/homework", "chapter_1.json")

正在识别: images (1).jpeg...


NameError: name 'content' is not defined

In [3]:
import Path
indices_file = Path("../data/hs_indices.pkl")

print(f"索引文件路径: {indices_file}")
print(f"文件是否存在: {indices_file.exists()}")
print(f"文件大小: {indices_file.stat().st_size / 1024:.2f} KB" if indices_file.exists() else "文件不存在")


ModuleNotFoundError: No module named 'Path'

## 8. 使用数据加载器（推荐方式）

使用 HighSchoolDataLoader 类来加载和分析数据

In [1]:
# 使用数据加载器（推荐方式）
import sys
sys.path.append('..')  # 添加项目根目录到路径

from high_school_data_loader import HighSchoolDataLoader

# 从索引文件加载
loader = HighSchoolDataLoader.load_indices(str(indices_file))

print("✅ 使用数据加载器加载成功！")
print(f"\n章节总数: {len(loader.chapter_index)}")
print(f"题目总数: {len(loader.question_index)}")
print(f"知识点总数: {len(loader.knowledge_index)}")

# 测试查询功能
print("\n测试查询功能:")
math_chapters = loader.get_chapters_by_subject("math")
print(f"数学科目章节数: {len(math_chapters)}")
if math_chapters:
    print(f"第一个章节: {math_chapters[0].get('title')}")

NameError: name 'indices_file' is not defined